In [10]:
import numpy as np

In [20]:
class Convolutional:
    def __init__(self, filters_n):
        self.filters_n = filters_n
        # Initialize filters with random values for learning
        self.filters = np.random.randn(filters_n, 3, 3) / 9
        self.last_input = None

    def forward(self, input):
        """Performs the convolution step with each kernel"""
        self.last_input = input
        h, w = input.shape
        output = np.zeros((h-2, w-2, self.filters_n))

        for x in range(w-2):
            for y in range(h-2):
                for z in range(self.filters_n):
                    # Apply the filter
                    output[y, x, z] = np.sum(input[y:y+3, x:x+3] * self.filters[z])

        return output

    def backward(self, output_error_gradient, learning_rate):
        """Performs the convolutional layer's backward pass"""
        # creating an empty array for storing the gradients for each filter
        filter_gradients = np.zeros(self.filters.shape)

        for x in range(self.last_input.shape[1] - 2):
            for y in range(self.last_input.shape[0] - 2):
                # calculating how much each filter contributed to the error
                for filter_index in range(self.filters_n):
                    filter_gradients[filter_index] += output_error_gradient[y, x, filter_index] * self.last_input[y:y+3, x:x+3]

        # gradient descent step
        self.filters -= learning_rate * filter_gradients

        return None

In [12]:
class MaxPool:
    def iterate_regions(self, image):
        h, w, _ = image.shape

        new_h = h // 2
        new_w = w // 2

        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i*2):(i*2+2), (j*2):(j*2+2)]
                yield im_region, i, j

    def forward(self, input):

        self.last_input = input

        h, w, num_filters = input.shape
        output = np.zeros((h//2, w//2, num_filters))

        for im_region, i, j in self.iterate_regions(input):
            output[i,j] = np.amax(im_region,axis=(0,1))

        return output

    def backprop(self, d_l_d_out):
        '''
        Performs a backward pass of the maxpool layer.
        Returns the loss gradient for this layer's inputs.
        - d_L_d_out is the loss gradient for this layer's outputs.
        '''
        d_l_d_input = np.zeros(self.last_input.shape)

        for im_region, i, j in self.iterate_regions(self.last_input):
            h, w, f = im_region.shape
            amax = np.amax(im_region, axis=(0,1))

            for i2 in range(h):
                for j2 in range(w):
                    for f2 in range(f):
                        #if the pixel was the max value, copy the gradient to it
                        if(im_region[i2,j2,f2] == amax[f2]):
                            d_l_d_input[i*2+i2, j*2+j2 ,f2] = d_l_d_out[i, j, f2]
                            break;
        return d_l_d_input

In [13]:
class Softmax:
    def __init__(self, input_len, nodes):
        # We divide by input_len to reduce the variance of our initial values
        self.weights = np.random.randn(input_len, nodes)/input_len
        self.biases = np.zeros(nodes)

    def forward(self, input):

        self.last_input_shape = input.shape

        input = input.flatten()
        self.last_input = input

        input_len, nodes = self.weights.shape

        totals = np.dot(input, self.weights) + self.biases
        self.last_totals = totals

        exp = np.exp(totals)
        return(exp/np.sum(exp, axis=0))

    def backprop(self, d_l_d_out, learn_rate):
        """
        Performs a backward pass of the softmax layer.
        Returns the loss gradient for this layers inputs.
        - d_L_d_out is the loss gradient for this layers outputs.
        """

        #We know only 1 element of d_l_d_out will be nonzero
        for i, gradient in enumerate(d_l_d_out):
            if(gradient == 0):
                continue

            #e^totals
            t_exp = np.exp(self.last_totals)

            #Sum of all e^totals
            S = np.sum(t_exp)

            #gradients of out[i] against totals
            d_out_d_t = -t_exp[i] * t_exp/ (S**2)
            d_out_d_t[i] = t_exp[i] * (S-t_exp[i]) /(S**2)

            # Gradients of totals against weights/biases/input
            d_t_d_w = self.last_input
            d_t_d_b = 1
            d_t_d_inputs = self.weights

            #Gradients of loss against totals
            d_l_d_t = gradient * d_out_d_t

            #Gradients of loss against weights/biases/input
            d_l_d_w = d_t_d_w[np.newaxis].T @ d_l_d_t[np.newaxis]
            d_l_d_b = d_l_d_t * d_t_d_b
            d_l_d_inputs = d_t_d_inputs @ d_l_d_t

            #update weights/biases
            self.weights -= learn_rate * d_l_d_w
            self.biases -= learn_rate * d_l_d_b
            return d_l_d_inputs.reshape(self.last_input_shape)

In [21]:
conv = Convolutional(8)
pool = MaxPool()
softmax = Softmax(31 * 31 * 8, 10)

def forward(image, label):
    # We transform the image from [0, 255] to [-0.5, 0.5] to make it easier
    # to work with. This is standard practice.

    out = conv.forward((image/255) - 0.5)
    out = pool.forward(out)
    out = softmax.forward(out)

    #calculate cross-entropy loss and accuracy
    loss = -np.log(out[label])
    acc = 1 if(np.argmax(out) == label) else 0

    return out, loss, acc


def train(im, label, lr=0.005):
    #forward
    out,loss,acc = forward(im, label)

    #calculate initial gradient
    gradient = np.zeros(10)
    gradient[label] = -1/out[label]


    #Backprop
    gradient = softmax.backprop(gradient, lr)
    gradient = pool.backprop(gradient)
    gradient = conv.backward(gradient, lr)

    return loss, acc

In [22]:
import os
import cv2

folder_path = "E:/dogs_vs_cats/train/train_small"
images = []
labels = []

# Load in the images
for filename in os.listdir(folder_path):
    # Use cv2 to read the image
    img = cv2.imread(os.path.join(folder_path, filename))
    img = cv2.resize(img, (64, 64))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    images.append(img)
    # Extract the label from the filename
    label = filename.split('.')[0]
    labels.append(1 if label == 'cat' else 0)

# Convert the lists to numpy arrays
images = np.array(images)
labels = np.array(labels)

# Shuffle the training data
permutation = np.random.permutation(len(images))
images = images[permutation]
labels = labels[permutation]

for epoch in range(10):
    print('----EPOCH %d ---'%(epoch+1))

    loss = 0
    num_correct = 0

    for i, (im, label) in enumerate(zip(images, labels)):

        # Print stats every 100 steps
        if(i > 0 and i % 100 == 99):
            print('[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %(i + 1, loss / 100, num_correct))

            loss = 0
            num_correct = 0
        l, acc = train(im, label)
        loss += l
        num_correct += acc

----EPOCH 1 ---
[Step 100] Past 100 steps: Average Loss 2.061 | Accuracy: 48%
[Step 200] Past 100 steps: Average Loss 1.421 | Accuracy: 55%
[Step 300] Past 100 steps: Average Loss 1.290 | Accuracy: 45%


KeyboardInterrupt: 

In [9]:
import os
import cv2

folder_path = "E:/dogs_vs_cats/train/train_small"
images = []
labels = []

# Load in the images
for filename in os.listdir(folder_path):
    # Use cv2 to read the image
    img = cv2.imread(os.path.join(folder_path, filename))
    img = cv2.resize(img, (64, 64))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    images.append(img)
    # Extract the label from the filename
    label = filename.split('.')[0]
    labels.append(1 if label == 'cat' else 0)

# Convert the lists to numpy arrays
images = np.array(images)
labels = np.array(labels)

# Shuffle the training data
permutation = np.random.permutation(len(images))
images = images[permutation]
labels = labels[permutation]

for epoch in range(10):
    print('----EPOCH %d ---'%(epoch+1))

    loss = 0
    num_correct = 0

    for i, (im, label) in enumerate(zip(images, labels)):

        # Print stats every 100 steps
        if(i > 0 and i % 100 == 99):
            print('[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %(i + 1, loss / 100, num_correct))

            loss = 0
            num_correct = 0
        l, acc = train(im, label)
        loss += l
        num_correct += acc

----EPOCH 1 ---
[Step 100] Past 100 steps: Average Loss 0.540 | Accuracy: 76%
[Step 200] Past 100 steps: Average Loss 0.584 | Accuracy: 70%
[Step 300] Past 100 steps: Average Loss 0.634 | Accuracy: 65%
[Step 400] Past 100 steps: Average Loss 0.555 | Accuracy: 73%
----EPOCH 2 ---
[Step 100] Past 100 steps: Average Loss 0.482 | Accuracy: 82%
[Step 200] Past 100 steps: Average Loss 0.511 | Accuracy: 71%
[Step 300] Past 100 steps: Average Loss 0.567 | Accuracy: 71%
[Step 400] Past 100 steps: Average Loss 0.477 | Accuracy: 81%
----EPOCH 3 ---
[Step 100] Past 100 steps: Average Loss 0.431 | Accuracy: 84%
[Step 200] Past 100 steps: Average Loss 0.449 | Accuracy: 76%
[Step 300] Past 100 steps: Average Loss 0.508 | Accuracy: 76%
[Step 400] Past 100 steps: Average Loss 0.415 | Accuracy: 86%
----EPOCH 4 ---
[Step 100] Past 100 steps: Average Loss 0.384 | Accuracy: 86%
[Step 200] Past 100 steps: Average Loss 0.394 | Accuracy: 83%
[Step 300] Past 100 steps: Average Loss 0.452 | Accuracy: 81%
[Step 